# 02 — KArSL QC and Split Verification

Read-only validation of existing QC, 500-class PASS-only benchmark and independent folds. Does not recreate or overwrite any reports.

## Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Dataset QC and benchmark checks

In [ ]:
from pathlib import Path
import pandas as pd
import os
B = Path(os.environ.get(
    "SIGNOVA_KARSL_ROOT",
    str(Path("/content") / "drive" / "MyDrive" / "SIGNOVA" / "KArSL")
))
R = B / "reports"
m = pd.read_csv(R / "metadata_arsl.csv", low_memory=False)
s = pd.read_csv(R / "split_manifest_3way.csv", low_memory=False, dtype={"sample_id": str, "sign_id": str})
pass_ids = set(m.loc[m["qc_status"].eq("PASS"), "sample_id"].astype(str))
active_ids = set(s["sample_id"].astype(str))
checks = {
    "Unique metadata IDs": m["sample_id"].is_unique,
    "Three folds": s["fold"].nunique() == 3,
    "All fold IDs in PASS": active_ids <= pass_ids,
    "No duplicate sample within fold": not s.duplicated(["fold", "sample_id"]).any(),
    "Consistent sample set per fold": all(set(g["sample_id"]) == active_ids for _, g in s.groupby("fold")),
    "Signer independent partitions": all(g.groupby("signer_id")["split"].nunique().max() == 1 for _, g in s.groupby("fold")),
    "Same classes in every partition": all(set(g["sign_id"]) == set(s["sign_id"]) for _, g in s.groupby(["fold", "split"]))
}
print("=== BENCHMARK SUMMARY ===")
print("Metadata samples:", len(m))
print("PASS samples:", len(pass_ids))
print("Unique benchmark samples:", len(active_ids))
print("Benchmark classes:", s["sign_id"].nunique())
print("QC status counts:")
print(m["qc_status"].value_counts().to_string())
print("=== VALIDATION ===")
for name, result in checks.items():
    print(name, "PASS" if result else "FAIL")
assert all(checks.values()), "Benchmark checks failed"

## Report presence and outstanding approvals

In [ ]:
from pathlib import Path
import pandas as pd
import os
B = Path(os.environ.get(
    "SIGNOVA_KARSL_ROOT",
    str(Path("/content") / "drive" / "MyDrive" / "SIGNOVA" / "KArSL")
))
R = B / "reports"
required = ["qc_report.md", "qc_flags.csv", "dataset_card.md", "handoff.md", "benchmark_v3_summary.md", "dataset_tests_v3.csv", "open_review_cases.csv", "class_mapping_v3.csv"]
print("=== REQUIRED REPORTS ===")
for f in required:
    print(f, "READY" if (R / f).is_file() else "MISSING")
assert all((R / f).is_file() for f in required)
tests = pd.read_csv(R / "dataset_tests_v3.csv", low_memory=False)
print("Recorded tests passing:", tests["passed"].astype(str).str.lower().eq("true").sum(), "/", len(tests))
print("Source license approval: PENDING")
print("Expert label review: PENDING")
print("Project lead approval: PENDING")
print("Original reports modified: NO")